# ⚡ Studio Video AI - Ponte de Conexão com Superplaca GPU (NVIDIA T4)
### Conecte a superplaca do Google Colab diretamente ao seu site / ferramenta

**Instruções de 1 Clique:**
1. Verifique se o ambiente está em **GPU T4** (Ambiente de execução -> Alterar tipo de ambiente -> GPU T4).
2. Clique no botão **Play ▶** abaixo.
3. Ele vai gerar um link **https://....trycloudflare.com**.
4. Copie o link, abra seu site, clique em **Conectar Motor GPU** e cole o link lá!
5. Pronto! Todo vídeo criado no site levará apenas **1 a 2 minutos**!

In [ ]:
# ==============================================================================
# ⚡ CONEXÃO DIRETA: MOTOR GPU NVIDIA T4 -> SUA FERRAMENTA WEB
# ==============================================================================
import os, sys, glob, site, time, re, subprocess, urllib.request, torch

# 🛑 1. TRAVA ANTI-CPU: Se a GPU não estiver ativa, ABORTA IMEDIATAMENTE!
if not torch.cuda.is_available():
    print('\n' + '!'*75)
    print('🛑 BLOQUEIO DE SEGURANÇA: NENHUMA GPU (PLACA DE VÍDEO) DETECTADA!')
    print('⚠️ O script foi interrompido para EVITAR que você perca horas no processador lento (CPU).')
    print('\n👉 COMO ATIVAR A GPU T4 GRATUITA NO GOOGLE COLAB:')
    print('   1. No menu superior, clique em "Ambiente de execução"')
    print('   2. Clique em "Alterar tipo de ambiente de execução"')
    print('   3. Em "Acelerador de hardware", selecione "T4 GPU"')
    print('   4. Clique em "Salvar" e aperte o Play novamente.')
    print('!'*75 + '\n')
    raise SystemExit('Script interrompido com sucesso: GPU obrigatória não conectada no ambiente.')

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print('\n' + '='*75)
print(f'⚡ SUPERPLACA DETECTADA COM SUCESSO: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
print('⚡ Modo Servidor de Alta Performance Ativado!')
print('='*75 + '\n')

print('📦 2. Verificando bibliotecas e ambiente...')
!pip install -q kornia facexlib yacs safetensors edge-tts pydub flask flask-cors

%cd /content
if not os.path.exists('/content/SadTalker'):
    !git clone -b v1.1 https://github.com/camenduru/SadTalker

%cd /content/SadTalker

# 3. Correções de compatibilidade total com NumPy 2.x
site_code = '''import numpy as np
import warnings
np.float = float
np.int = int
np.bool = bool
if not hasattr(np, 'VisibleDeprecationWarning'):
    np.VisibleDeprecationWarning = UserWarning
'''
for sp in site.getsitepackages():
    try:
        with open(os.path.join(sp, 'sitecustomize.py'), 'w') as f:
            f.write(site_code)
    except:
        pass

for root, dirs, files_list in os.walk('/content/SadTalker'):
    for f in files_list:
        if f.endswith('.py'):
            fpath = os.path.join(root, f)
            try:
                with open(fpath, 'r', encoding='utf-8') as fp:
                    txt = fp.read()
                new_txt = txt.replace('np.float,', 'float,').replace('np.float)', 'float)').replace('np.VisibleDeprecationWarning', 'UserWarning')
                if new_txt != txt:
                    with open(fpath, 'w', encoding='utf-8') as fp:
                        fp.write(new_txt)
            except:
                pass

fixed_preprocess_py = '''import numpy as np
from PIL import Image
import warnings

if not hasattr(np, 'VisibleDeprecationWarning'):
    np.VisibleDeprecationWarning = UserWarning
warnings.filterwarnings('ignore')

def POS(xp, x):
    npts = xp.shape[1]
    A = np.zeros([2*npts, 8])
    A[0:2*npts-1:2, 0:3] = x.transpose()
    A[0:2*npts-1:2, 3] = 1
    A[1:2*npts:2, 4:7] = x.transpose()
    A[1:2*npts:2, 7] = 1
    b = np.reshape(xp.transpose(), [2*npts, 1])
    k, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    R1 = k[0:3]
    R2 = k[4:7]
    sTx = float(np.squeeze(k[3]))
    sTy = float(np.squeeze(k[7]))
    s = float((np.linalg.norm(R1) + np.linalg.norm(R2))/2)
    t = np.array([sTx, sTy], dtype=np.float32)
    return t, s

def resize_n_crop_img(img, lm, t, s, target_size=224., mask=None):
    w0, h0 = img.size
    w = int(w0*s)
    h = int(h0*s)
    t0 = float(np.squeeze(t[0]))
    t1 = float(np.squeeze(t[1]))
    left = int(w/2 - target_size/2 + (t0 - w0/2)*s)
    right = int(left + target_size)
    up = int(h/2 - target_size/2 + (h0/2 - t1)*s)
    below = int(up + target_size)

    img = img.resize((w, h), resample=Image.BICUBIC)
    img = img.crop((left, up, right, below))

    if mask is not None:
        mask = mask.resize((w, h), resample=Image.BICUBIC)
        mask = mask.crop((left, up, right, below))

    lm = np.stack([lm[:, 0] - t0 + w0/2, lm[:, 1] - t1 + h0/2], axis=1)*s
    lm = lm - np.reshape(np.array([(w/2 - target_size/2), (h/2-target_size/2)]), [1, 2])

    return img, lm, mask

def extract_5p(lm):
    lm_idx = np.array([31, 37, 40, 43, 46, 49, 55]) - 1
    lm5p = np.stack([lm[lm_idx[0], :], np.mean(lm[lm_idx[[1, 2]], :], 0), np.mean(
        lm[lm_idx[[3, 4]], :], 0), lm[lm_idx[5], :], lm[lm_idx[6], :]], axis=0)
    lm5p = lm5p[[1, 2, 0, 3, 4], :]
    return lm5p

def align_img(img, lm, lm3D, mask=None, target_size=224., rescale_factor=102.):
    w0, h0 = img.size
    if lm.shape[0] != 5:
        lm5p = extract_5p(lm)
    else:
        lm5p = lm

    t, s = POS(lm5p.transpose(), lm3D.transpose())
    s = rescale_factor/s

    img_new, lm_new, mask_new = resize_n_crop_img(img, lm, t, s, target_size=target_size, mask=mask)
    t0 = float(np.squeeze(t[0]))
    t1 = float(np.squeeze(t[1]))
    trans_params = np.array([float(w0), float(h0), float(s), t0, t1], dtype=np.float32)

    return trans_params, img_new, lm_new, mask_new
'''
with open('/content/SadTalker/src/face3d/util/preprocess.py', 'w') as f:
    f.write(fixed_preprocess_py)

utils_preprocess = '/content/SadTalker/src/utils/preprocess.py'
with open(utils_preprocess, 'r') as f:
    u_content = f.read()
u_content = u_content.replace('[float(item) for item in np.hsplit(trans_params, 5)]', '[float(np.squeeze(item)) for item in np.hsplit(trans_params, 5)]')
with open(utils_preprocess, 'w') as f:
    f.write(u_content)

!curl -s -L -o /content/SadTalker/src/utils/face_enhancer.py https://raw.githubusercontent.com/camenduru/SadTalker/v1.1/src/utils/face_enhancer.py
enhancer_file = '/content/SadTalker/src/utils/face_enhancer.py'
with open(enhancer_file, 'r') as f:
    enh_content = f.read()
enh_content = enh_content.replace('from gfpgan import GFPGANer', 'GFPGANer = None')
with open(enhancer_file, 'w') as f:
    f.write(enh_content)

# 4. Baixa/Verifica pesos neurais 512px do SadTalker
!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_256.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_256.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_512.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_512.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00109-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00109-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00229-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00229-model.pth.tar

# Baixa assets padrão
!curl -s -L -o /content/personagem_maconaria.jpg https://raw.githubusercontent.com/grego921/studio-video-ai/main/personagem_maconaria.jpg
!curl -s -L -o /content/fala_maconica.mp3 https://raw.githubusercontent.com/grego921/studio-video-ai/main/fala_maconica.mp3

# 5. Baixa a API da ponte e o Cloudflare Tunnel
!curl -s -L -o /content/colab_api.py https://raw.githubusercontent.com/grego921/studio-video-ai/main/colab_api.py
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Mata instâncias antigas se houver
!pkill -f colab_api.py
!pkill -f cloudflared

print('\n🚀 Iniciando Servidor API da GPU...')
api_proc = subprocess.Popen(['python', '/content/colab_api.py'], stdout=open('/content/api.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(3)

print('🌐 Abrindo Túnel Cloudflare para a Internet...')
tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'], stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists('/content/tunnel.log'):
        with open('/content/tunnel.log') as f:
            content = f.read()
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if m:
            tunnel_url = m.group(0)
            break

if tunnel_url:
    print('\n' + '='*75)
    print('⚡ MOTOR GPU CONECTADO COM SUCESSO NA SUPERPLACA NVIDIA T4!')
    print('='*75)
    print('\n👉 COPIE ESTA URL E COLE NA SUA FERRAMENTA (SITE):\n')
    print(f'   {tunnel_url}\n')
    print('='*75)
    print('🟢 COMO USAR NO SEU SITE:')
    print('1. Abra sua ferramenta no navegador.')
    print('2. Clique no botão "Conectar Motor GPU".')
    print(f'3. Cole a URL acima ({tunnel_url}) e clique em "Salvar e Conectar".')
    print('4. O indicador vai ficar verde: 🟢 Motor GPU Ativo (NVIDIA T4).')
    print('5. Agora qualquer vídeo gerado no site será processado na superplaca')
    print('   em apenas 1 a 2 minutos!')
    print('='*75 + '\n')
    print('⏳ Mantenha esta célula ativa no Colab enquanto estiver usando seu site.')
    try:
        tunnel_proc.wait()
    except KeyboardInterrupt:
        print('Servidor finalizado pelo usuário.')
else:
    print('❌ Falha ao obter URL pública do túnel. Verifique /content/tunnel.log')
